# FASE 2 — Limpeza e Pré-processamento dos Dados

Este notebook documenta o processo de limpeza e preparação do dataset da Netflix, corrigindo inconsistências identificadas na Fase 1 (Diagnóstico).

In [11]:
import pandas as pd
import numpy as np
import os

# Configurações de exibição
pd.set_option('display.max_columns', None)

## 1. Carregamento dos Dados Brutos

In [12]:
df = pd.read_csv('../dados_brutos/netflix_raw.csv')
print(f"Formato original: {df.shape}")
df.head()

Formato original: (8820, 10)


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1287,Movie,Classmates Minus,Huang Hsin-Yao,Taiwan,2/20/2021,2020,NaN,123 min,"Comedies, Dramas, Independent Movies"
1,s5310,Movie,Sohni Mahiwal,"Latif Faiziyev, Umesh Mehra",India,9/1/2017,1984,TV-14,160 min,"Dramas, International Movies, Romantic Movies"
2,s6753,Movie,Filosofi Kopi The Movie,Angga Dwimas Sasongko,Indonesia,10/11/2018,2015,TV-PG,118 min,"Dramas, International Movies"
3,s3037,TV Show,Jamtara - Sabka Number Ayega,Soumendra Padhi,NaN,1/10/2020,2020,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Dramas"
4,s4339,Movie,Don't Go Breaking My Heart 2,NaN,Hong Kong,12/1/2018,2014,NaN,113 min,"Comedies, International Movies, Romantic Movies"


## 2. Tratamento de Missing Values (Placeholders)

Substituindo textos que indicam valores ausentes por `NaN` real do Pandas.

In [13]:
placeholders = ['Not Given', '???', 'NULL', 'N/A', '', 'Unknown']
df.replace(placeholders, np.nan, inplace=True)
print(f"Nulos após tratamento de placeholders:\n{df.isnull().sum()}")

Nulos após tratamento de placeholders:
show_id            0
type               1
title              1
director        3533
country         1559
date_added      1320
release_year       0
rating          1322
duration           0
listed_in          0
dtype: int64


## 3. Padronização Textual

Removendo espaços extras (trim) e garantindo que strings vazias ou 'nan' sejam tratadas como nulas.

In [14]:
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].astype(str).str.strip()
        df.loc[df[col] == 'nan', col] = np.nan

## 4. Correção Estrutural (Colunas Deslocadas)

Corrigindo casos onde a classificação indicativa (rating) foi parar na coluna de tipo (`type`).

In [15]:
valid_types = ['Movie', 'TV Show']
mask_shifted = df['type'].notna() & ~df['type'].str.title().isin(valid_types)

# Mover rating do 'type' para a coluna 'rating' se estiver nulo
df.loc[mask_shifted & df['rating'].isna(), 'rating'] = df.loc[mask_shifted, 'type']

# Inferir o tipo correto com base na duração
def infer_type(duration):
    if pd.isna(duration): return np.nan
    d = str(duration).lower()
    return 'TV Show' if 'season' in d else 'Movie'

df.loc[mask_shifted, 'type'] = df.loc[mask_shifted, 'duration'].apply(infer_type)

# Padronizar 'type' de forma robusta
def standardize_type(val):
    if pd.isna(val): return val
    v = str(val).strip().lower()
    if 'movie' in v: return 'Movie'
    if 'tv show' in v or 'tv' in v: return 'TV Show'
    return val

df['type'] = df['type'].apply(standardize_type)
df.loc[~df['type'].isin(valid_types), 'type'] = np.nan
print(f"Tipos únicos após correção: {df['type'].unique()}")

Tipos únicos após correção: <StringArray>
['Movie', 'TV Show', nan]
Length: 3, dtype: str


## 5. Tratamento de Duplicados

Removendo registros exatos e IDs repetidos.

In [16]:
before = len(df)
df.drop_duplicates(subset=[c for c in df.columns if c != 'show_id'], inplace=True)
df.drop_duplicates(subset='show_id', inplace=True)
after = len(df)
print(f"Registros removidos: {before - after}")

Registros removidos: 30


## 6. Padronização de Unidades e Tipos

Padronizando a coluna `duration` e convertendo datas.

In [17]:
def standardize_duration(d):
    if pd.isna(d): return d
    d = str(d).lower()
    num = ''.join(filter(str.isdigit, d))
    if 'min' in d or 'minute' in d or 'm' in d:
        return f"{num} min"
    if 'season' in d:
        return f"{num} Season" if num == "1" else f"{num} Seasons"
    return d

df['duration'] = df['duration'].apply(standardize_duration)
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
df['release_year'] = pd.to_numeric(df['release_year'], errors='coerce')
df.info()

<class 'pandas.DataFrame'>
Index: 8790 entries, 0 to 8819
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   show_id       8790 non-null   str           
 1   type          8789 non-null   str           
 2   title         8789 non-null   str           
 3   director      5266 non-null   str           
 4   country       7234 non-null   str           
 5   date_added    7476 non-null   datetime64[us]
 6   release_year  8775 non-null   float64       
 7   rating        7852 non-null   str           
 8   duration      8790 non-null   str           
 9   listed_in     8790 non-null   str           
dtypes: datetime64[us](1), float64(1), str(8)
memory usage: 755.4 KB


## 7. Verificação Lógica

Identificando inconsistências temporais (data de adição anterior ao lançamento).

In [18]:
invalid = df[df['date_added'].dt.year < df['release_year']]
print(f"Registros inconsistentes encontrados: {len(invalid)}")

Registros inconsistentes encontrados: 34


## 8. Salvando o Resultado

Exportando para a pasta de dados intermediários.

In [19]:
os.makedirs('../dados_intermediarios', exist_ok=True)
df.to_csv('../dados_intermediarios/netflix_clean.csv', index=False)
print("Arquivo salvo com sucesso!")

Arquivo salvo com sucesso!
